In [1]:
# ------------------------------
# Notebook completo: Demo encuesta interactiva con recomendaciones mejoradas
# ------------------------------
import os, glob, re
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

# ------------------------------
# Clase y loaders (diagnóstico y CSV)
# ------------------------------
class EncuestaPregunta:
    def __init__(self, seccion, texto, tipo="Texto", opciones_raw=""):
        self.seccion = seccion or ""
        self.texto = texto or ""
        self.tipo = tipo or "Texto"
        self.opciones_raw = opciones_raw or ""

def load_from_xlsx_debug(path):
    """Lectura diagnóstica: detecta encabezado con 'Pregunta', construye lista de preguntas."""
    print("Leyendo archivo:", path)
    raw = pd.read_excel(path, sheet_name=0, header=None, engine="openpyxl")
    print("Dimensiones (raw):", raw.shape)
    display(raw.iloc[:12, :7])
    header_row = None
    for i in range(min(40, raw.shape[0])):
        row_text = " ".join([str(x) for x in raw.iloc[i].values if pd.notna(x)])
        if re.search(r'pregunta', row_text, flags=re.I):
            header_row = i
            break
    print("Fila detectada como encabezado (index):", header_row)
    if header_row is None:
        df = pd.read_excel(path, sheet_name=0, header=0, engine="openpyxl")
    else:
        df = pd.read_excel(path, sheet_name=0, header=header_row, engine="openpyxl")
    print("Dimensiones (df con header):", df.shape)
    cols = list(df.columns)
    col_id = None; col_preg = None; col_opts = None
    for c in cols:
        cn = str(c).strip().lower()
        if cn == 'id' or cn.startswith('id'): col_id = c
        if 'pregunta' in cn: col_preg = c
        if 'opcion' in cn or 'respuesta' in cn or 'posibl' in cn: col_opts = c
    if col_preg is None and len(cols) >= 2:
        col_preg = cols[1]; print("Fallback: usar segunda columna como Pregunta:", col_preg)
    if col_id is None:
        col_id = cols[0]; print("Fallback: usar primera columna como ID:", col_id)
    if col_opts is None and len(cols) >= 3:
        col_opts = cols[2]; print("Fallback: usar tercera columna como Opciones:", col_opts)

    ids = pd.to_numeric(df[col_id], errors='coerce')
    df_ids = df[~ids.isna()].copy()
    df_ids[col_id] = ids.dropna().astype(int)
    if df_ids.shape[0] == 0:
        non_empty = df[df[col_preg].notna() & (df[col_preg].astype(str).str.strip() != "")]
        df_unique = non_empty.reset_index(drop=True)
    else:
        df_unique = df_ids.drop_duplicates(subset=[col_id], keep='first').sort_values(by=col_id)

    preguntas = []
    for _, r in df_unique.iterrows():
        seccion = r.get('Seccion','') if 'Seccion' in r.index else ""
        texto = str(r.get(col_preg,"")).strip()
        opciones_raw = str(r.get(col_opts,"")).strip() if (col_opts in r.index) else ""
        tipo = "Seleccion" if '|' in opciones_raw else "Texto"
        preguntas.append({"seccion": seccion, "texto": texto, "tipo": tipo, "opciones": opciones_raw})
    print("Preguntas detectadas (len):", len(preguntas))
    return preguntas

def load_from_csv(path):
    df = pd.read_csv(path, dtype=str).fillna("")
    preguntas = []
    cols = list(df.columns)
    col_sec = cols[0] if len(cols) >= 1 else None
    col_preg = cols[1] if len(cols) >= 2 else cols[0]
    col_tipo = cols[2] if len(cols) >= 3 else None
    col_opts = cols[3] if len(cols) >= 4 else None
    for _, row in df.iterrows():
        sec = row.get(col_sec,"") if col_sec in row.index else ""
        preg = row.get(col_preg,"")
        tipo = row.get(col_tipo,"Texto") if col_tipo in row.index else "Texto"
        opts = row.get(col_opts,"") if col_opts in row.index else ""
        preguntas.append(EncuestaPregunta(sec, preg, tipo, opts))
    return preguntas

# ------------------------------
# Recomendador: mapeo de temas -> recursos + reglas
# ------------------------------
# Mapa simple de temas con recursos (puedes editar los textos/URL)
TOPIC_RESOURCES = {
    "matematicas": [
        ("Álgebra básica - Guía PDF", "Ficha de ejercicios y teoría básica"),
        ("Playlist: Matemáticas universitarias (30 videos)", "Videos cortos por tema"),
        ("Curso gratis: Fundamentos de cálculo (MOOC)", "Curso autodidacta con ejercicios")
    ],
    "lectura": [
        ("Taller comprensión lectora - Manual", "Técnicas y ejercicios prácticos"),
        ("Guía de resúmenes y esquemas", "Cómo sintetizar textos académicos"),
        ("Curso: Lectura crítica y análisis", "Estrategias para estudio universitario")
    ],
    "programacion": [
        ("Curso Python para principiantes", "Fundamentos, variables, estructuras"),
        ("Repo: Proyectos prácticos para aprender programación", "Proyectos guiados"),
        ("Guía: Buenas prácticas y debugging", "Consejos para depurar código")
    ],
    "finanzas": [
        ("Curso básico de finanzas personales", "Ahorro, deuda y presupuesto"),
        ("Plantilla de presupuesto mensual (Excel)", "Ejemplo práctico para estudiantes"),
        ("Lectura: Cómo manejar tarjetas y créditos", "Consejos y riesgos")
    ],
    "motivacion": [
        ("Técnicas de estudio y planificación", "Organiza tu tiempo y evita procrastinar"),
        ("Guía de hábitos productivos", "Rutinas para mejorar rendimiento"),
        ("Taller de gestión emocional para estudiantes", "Recursos de bienestar")
    ]
}

# Palabras clave por tema (heurística)
TOPIC_KEYWORDS = {
    "matematicas": ["matem", "álgebra", "calculo", "aritmética", "estadística"],
    "lectura": ["lectura", "comprensión", "texto", "resumen", "lect"],
    "programacion": ["program", "python", "java", "algoritm", "codigo", "comput"],
    "finanzas": ["dinero", "deuda", "presupuesto", "finanzas", "ahorro"],
    "motivacion": ["motiv", "ansiedad", "tiempo", "procrastin", "estres"]
}

def infer_topics_from_text(text):
    text = str(text).lower()
    hits = {}
    for topic, kws in TOPIC_KEYWORDS.items():
        for kw in kws:
            if kw in text:
                hits[topic] = hits.get(topic, 0) + 1
    return hits  # dict topic -> score (counts)

def severity_from_value(val):
    """Asignar severidad (0-3) basada en la respuesta textual"""
    if val is None: return 0
    s = str(val).strip().lower()
    if s == "": return 0
    # detectores de bajo rendimiento / alerta
    low_tokens = ["bajo","malo","nunca","poco","pocas","1","2","muy bajo","deficiente","pobre"]
    med_tokens = ["medio","3","regular","ocasionalmente"]
    high_tokens = ["alto","4","5","si","mucho","siempre","bueno","bien"]
    for t in low_tokens:
        if t in s: return 3
    for t in med_tokens:
        if t in s: return 2
    for t in high_tokens:
        if t in s: return 1
    # si es selección múltiple y la respuesta es una lista con indicadores
    if isinstance(val, (list, tuple, set)):
        # si hay items que parezcan problema, vuelta alta severidad
        for item in val:
            it = str(item).lower()
            for t in low_tokens:
                if t in it: return 3
    return 1

def generate_recommendations(preguntas_global, respuestas_guardadas, top_n=5):
    """Genera recomendaciones basadas en respuestas guardadas."""
    # acumuladores por tema
    topic_signals = {}  # topic -> [ (severity, reason_text) ]
    for idx, resp in respuestas_guardadas.items():
        p = preguntas_global[idx]
        # infer topics from question text and section
        hits = infer_topics_from_text(p.texto + " " + p.seccion)
        # if no hits, try scanning options text
        if not hits and p.opciones_raw:
            hits = infer_topics_from_text(p.opciones_raw)
        # compute severity
        sev = severity_from_value(resp)
        # for each hit, add a signal
        if hits:
            for t, score in hits.items():
                reason = f"Pregunta {idx+1}: '{p.texto[:80]}' → respuesta: {resp}"
                topic_signals.setdefault(t, []).append((sev, reason))
        else:
            # fallback: if question text contains numbers or explicit words
            # try simple heuristics for "matem" existence
            if "matem" in p.texto.lower() or "matem" in p.seccion.lower():
                reason = f"Pregunta {idx+1}: '{p.texto[:80]}' → respuesta: {resp}"
                topic_signals.setdefault("matematicas", []).append((sev, reason))
            else:
                # unknown topic: map to motivacion as fallback
                reason = f"Pregunta {idx+1}: '{p.texto[:80]}' → respuesta: {resp}"
                topic_signals.setdefault("motivacion", []).append((sev, reason))

    # produce scored list of recommendations
    rec_list = []
    for topic, signals in topic_signals.items():
        # score: sum of severities + count
        score = sum(s for s,_ in signals) + len(signals)*0.5
        # pick resources
        resources = TOPIC_RESOURCES.get(topic, [])
        rec_list.append((score, topic, signals, resources))

    # sort by score desc
    rec_list.sort(reverse=True, key=lambda x: x[0])

    # build readable output
    out = []
    for score, topic, signals, resources in rec_list[:top_n]:
        reasons = "\n".join([f"- severidad {s}: {r}" for s,r in signals])
        out.append({
            "topic": topic,
            "score": score,
            "reasons": reasons,
            "resources": resources
        })
    return out

# ------------------------------
# Buscar archivo automáticamente (mismas rutas que antes)
# ------------------------------
TARGET_NAMES = ["EncuestaAdmisiones.xlsx", "EncuestaAdmisiones.csv", "encuesta_admisiones.xlsx", "encuesta_demo.csv"]
found_paths = []
candidates = [os.path.join(os.getcwd(), TARGET_NAMES[0]), os.path.join("/mnt/data", TARGET_NAMES[0]), os.path.join(os.path.expanduser("~"), TARGET_NAMES[0])]

for root in ["/mnt", os.getcwd(), os.path.expanduser("~")]:
    try:
        for t in TARGET_NAMES:
            for path in glob.glob(os.path.join(root, "**", t), recursive=True):
                found_paths.append(path)
                if len(found_paths) >= 50: break
    except Exception:
        pass

for c in candidates:
    if os.path.exists(c) and c not in found_paths:
        found_paths.insert(0, c)

found_paths = list(dict.fromkeys(found_paths))

preguntas_global = []
detected_path = None
if found_paths:
    detected_path = found_paths[0]
    print("Archivo encontrado automáticamente:", detected_path)
    if detected_path.lower().endswith(".xlsx"):
        pregs = load_from_xlsx_debug(detected_path)
        preguntas_global = [EncuestaPregunta(d.get('seccion',''), d.get('texto',''), d.get('tipo','Texto'), d.get('opciones','')) for d in pregs]
    else:
        preguntas_global = load_from_csv(detected_path)
    print("Asignado preguntas_global con", len(preguntas_global), "elementos.")
else:
    print("No se encontró archivo automáticamente. Puedes subirlo manualmente.")
    upload = widgets.FileUpload(accept=".xlsx, .csv", multiple=False)
    btn_upload = widgets.Button(description="Guardar archivo subido en cwd")
    out_upload = widgets.Output()
    def on_upload_save(b):
        with out_upload:
            clear_output()
            if not upload.value:
                print("No has subido ningún archivo aún.")
                return
            fname = list(upload.value.keys())[0]
            content = upload.value[fname]['content']
            save_path = os.path.join(os.getcwd(), fname)
            with open(save_path, "wb") as f:
                f.write(content)
            print("Archivo guardado en:", save_path)
            print("Reejecuta esta celda para que lo detecte automáticamente.")
    btn_upload.on_click(on_upload_save)
    display(widgets.HTML("<b>Sube tu EncuestaAdmisiones.xlsx o CSV aquí si no se detectó automáticamente:</b>"))
    display(upload, btn_upload, out_upload)

# ------------------------------
# UI: construye opciones y widgets (soporta multiple)
# ------------------------------
respuestas_guardadas = {}

def build_options_from_preguntas():
    options = []
    for i, p in enumerate(preguntas_global):
        label = f"{i+1}. [{p.seccion}] {p.texto[:120]}"
        options.append((label, i))
    return options

list_box = widgets.Select(options=build_options_from_preguntas(), rows=15, layout=widgets.Layout(width='50%'))
resp_area = widgets.VBox([], layout=widgets.Layout(border='1px solid gray', padding='10px', width='100%'))
btn_prev = widgets.Button(description="⟨ Anterior")
btn_next = widgets.Button(description="Siguiente ⟩")
lbl_index = widgets.Label(value="Selecciona una pregunta")

def _get_option_values():
    try:
        return [opt[1] for opt in list_box.options]
    except Exception:
        return []

def go_prev(b):
    vals = _get_option_values()
    if not vals: return
    cur = list_box.value
    try: idx = vals.index(cur)
    except ValueError: idx = 0
    new_idx = max(0, idx - 1)
    list_box.value = vals[new_idx]

def go_next(b):
    vals = _get_option_values()
    if not vals: return
    cur = list_box.value
    try: idx = vals.index(cur)
    except ValueError: idx = 0
    new_idx = min(len(vals)-1, idx + 1)
    list_box.value = vals[new_idx]

btn_prev.on_click(go_prev)
btn_next.on_click(go_next)

def render_selected(change):
    val = change.get('new', None)
    if val is None:
        resp_area.children = [widgets.HTML("<i>No hay preguntas para mostrar.</i>")]
        lbl_index.value = "Selecciona una pregunta"
        return
    idx = int(val)
    p = preguntas_global[idx]
    title = widgets.HTML(f"<h4>{idx+1}. {p.texto}</h4><b>Sección:</b> {p.seccion}  &nbsp; <b>Tipo:</b> {p.tipo}")
    tipo_lower = str(getattr(p, "tipo", "")).lower()
    opts_raw = str(getattr(p, "opciones_raw", "")).strip()

    # Detectar si debe ser SelectMultiple:
    # Condiciones: tipo contiene "multiple" OR opciones incluyen la marca [M]
    is_multiple = False
    if "multiple" in tipo_lower or "[m]" in opts_raw.lower():
        is_multiple = True

    # Si hay opciones separadas por '|' y no es multiple explícito -> RadioButtons (elección única)
    if "|" in opts_raw and not is_multiple:
        opts = [o.strip() for o in opts_raw.split("|") if o.strip()]
        control = widgets.RadioButtons(options=opts)
    elif "|" in opts_raw and is_multiple:
        opts = [o.strip() for o in opts_raw.replace("[M]","").split("|") if o.strip()]
        control = widgets.SelectMultiple(options=opts, rows=min(6, len(opts)))
    elif "escala" in tipo_lower:
        opts = [o.strip() for o in opts_raw.split("|") if o.strip()]
        control = widgets.ToggleButtons(options=opts or ["1","2","3","4","5"])
    else:
        # Sin opciones: texto libre
        control = widgets.Textarea(placeholder="Escribe la respuesta aquí", layout=widgets.Layout(width='100%', height='120px'))

    # cargar valor guardado si existe (si fue SelectMultiple, cargar como tupla/lista)
    if idx in respuestas_guardadas:
        try:
            saved = respuestas_guardadas[idx]
            if isinstance(control, widgets.SelectMultiple):
                # espera lista/tupla
                control.value = tuple(saved) if isinstance(saved, (list,tuple)) else (saved,)
            else:
                control.value = saved
        except Exception:
            pass

    resp_area.children = [title, control]
    lbl_index.value = f"Pregunta {idx+1} / {len(preguntas_global)}"

list_box.observe(lambda ch: render_selected(ch), names='value')

display(widgets.HBox([list_box, resp_area]))
display(widgets.HBox([btn_prev, lbl_index, btn_next]))

vals = _get_option_values()
if vals:
    list_box.value = vals[0]
else:
    resp_area.children = [widgets.HTML("<i>No hay preguntas para mostrar. Carga o sube el archivo y reejecuta esta celda.</i>")]

# ------------------------------
# Controles: guardar, estadísticas, recomendaciones, export
# ------------------------------
out_stats = widgets.Output()
out_rec = widgets.Output()
out_export = widgets.Output()

btn_save = widgets.Button(description="Guardar respuesta (pregunta actual)", button_style='success')
btn_stats = widgets.Button(description="Ver estadísticas", button_style='info')
btn_rec = widgets.Button(description="Generar recomendaciones", button_style='warning')
btn_export = widgets.Button(description="Exportar respuestas a CSV", button_style='')

lbl_save = widgets.HTML("")

def on_save(b):
    if list_box.value is None:
        lbl_save.value = "<span style='color:red'>Selecciona una pregunta primero.</span>"
        return
    idx = int(list_box.value)
    if len(resp_area.children) < 2:
        lbl_save.value = "<span style='color:orange'>No hay control de respuesta visible.</span>"
        return
    control = resp_area.children[1]
    val = ""
    try:
        if isinstance(control, widgets.SelectMultiple):
            val = tuple(control.value)
        else:
            val = control.value if hasattr(control, "value") else str(control)
    except Exception:
        val = str(getattr(control, "value", ""))
    respuestas_guardadas[idx] = val
    lbl_save.value = f"<span style='color:green'>Respuesta guardada (pregunta {idx+1}).</span>"

def on_stats(b):
    with out_stats:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas guardadas.")
            return
        rows = []
        for idx, val in respuestas_guardadas.items():
            p = preguntas_global[idx]
            rows.append({"Índice": idx+1, "Sección": p.seccion, "Pregunta": p.texto, "Respuesta": val})
        df = pd.DataFrame(rows)
        display(df)
        print("\nResumen por pregunta (conteos):")
        display(df.groupby(["Pregunta","Respuesta"]).size().reset_index(name="Conteo"))

def on_rec(b):
    with out_rec:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas guardadas para evaluar.")
            return
        recs = generate_recommendations(preguntas_global, respuestas_guardadas, top_n=6)
        if not recs:
            print("No se generaron recomendaciones (sin señales relevantes).")
            return
        for r in recs:
            print(f"\n=== Tema: {r['topic'].upper()}  (score {r['score']:.1f}) ===")
            print("Razones que activaron la recomendación:")
            print(r['reasons'])
            print("\nRecursos sugeridos:")
            for title, desc in r['resources']:
                print(f" - {title}: {desc}")

def on_export(b):
    with out_export:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas para exportar.")
            return
        rows = []
        for idx, val in respuestas_guardadas.items():
            p = preguntas_global[idx]
            rows.append({"Índice": idx+1, "Sección": p.seccion, "Pregunta": p.texto, "Respuesta": val})
        df = pd.DataFrame(rows)
        path = "respuestas_exportadas.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Respuestas exportadas a: {os.path.abspath(path)}")

btn_save.on_click(on_save)
btn_stats.on_click(on_stats)
btn_rec.on_click(on_rec)
btn_export.on_click(on_export)

display(widgets.HBox([btn_save, btn_stats, btn_rec, btn_export, lbl_save]))
display(out_stats, out_rec, out_export)

print("\nNotebook listo. Usa la lista a la izquierda para seleccionar preguntas, responde y pulsa 'Guardar respuesta'.")


Archivo encontrado automáticamente: c:\Users\Ximena\Desktop\ModelosComputacionales_Tinoco\ProyectoIS\EncuestaAdmisiones.xlsx
Leyendo archivo: c:\Users\Ximena\Desktop\ModelosComputacionales_Tinoco\ProyectoIS\EncuestaAdmisiones.xlsx
Dimensiones (raw): (462, 8)


,0,1,2,3,4,5,6
0,UNIVERSIDAD AUTONOMA DE COAHUILA,NaN,NaN,NaN,NaN,NaN,NaN
1,Dirección de Asuntos Académicos,NaN,NaN,NaN,NaN,NaN,NaN
2,Encuesta estudiantes de reingreso,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID,Pregunta,Posibles respuestas,NaN,NaN,NaN,NaN
5,INFORMACIÓN PERSONAL,NaN,NaN,NaN,NaN,NaN,NaN
6,1,Nombre,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,Matricula,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Fila detectada como encabezado (index): 4
Dimensiones (df con header): (457, 8)
Preguntas detectadas (len): 59
Asignado preguntas_global con 59 elementos.


Output()

Output()

Output()


Notebook listo. Usa la lista a la izquierda para seleccionar preguntas, responde y pulsa 'Guardar respuesta'.
